<!-- In this notebook (extension of exp1) we will load the dataset once, split it deterministically into 80:10:10 (train:val:test), train with validation, use early stopping, save the best model, and evaluate on test set -->

In [ ]:
# ===============================
# 0. IMPORTS
# ===============================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, Subset

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import DataStructs
import numpy as np

from transformers import BertTokenizer, BertModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import os
import random

# ===============================
# 1. DEVICE
# ===============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ===============================
# 2. SMILES → GRAPH
# ===============================
def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    x = []
    for atom in mol.GetAtoms():
        x.append([
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetFormalCharge(),
            int(atom.GetHybridization())
        ])
    x = torch.tensor(x, dtype=torch.float)

    edge_index = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index.append([i, j])
        edge_index.append([j, i])

    if len(edge_index) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    return Data(x=x, edge_index=edge_index)

# ===============================
# 3. DOMAIN FEATURES
# ===============================
def morgan_fp(smiles, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return torch.tensor(arr, dtype=torch.float)

AA = 'ACDEFGHIKLMNPQRSTVWY'

def protein_aac(seq):
    seq = seq.upper()
    if len(seq) == 0:
        return torch.zeros((len(AA),), dtype=torch.float)
    return torch.tensor([seq.count(a)/len(seq) for a in AA], dtype=torch.float)

# ===============================
# 4. DATASET
# ===============================
class CPIDataset(Dataset):
    def __init__(self, file_path):
        self.samples = []
        with open(file_path) as f:
            for line in f:
                s, p, y = line.strip().split()
                self.samples.append((s, p, int(y)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        smiles, protein, label = self.samples[idx]

        data = smiles_to_graph(smiles)
        data.fp = morgan_fp(smiles).unsqueeze(0)
        data.aac = protein_aac(protein).unsqueeze(0)
        data.protein_seq = protein
        data.y = torch.tensor(label, dtype=torch.float)

        return data

# ===============================
# 5. MODEL COMPONENTS
# ===============================
class CompoundGNN(nn.Module):
    def __init__(self, node_dim=4, hidden=128):
        super().__init__()
        self.conv1 = GCNConv(node_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        return global_mean_pool(x, batch)

class ProteinEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokenizer = BertTokenizer.from_pretrained(
            "Rostlab/prot_bert", do_lower_case=False
        )
        self.model = BertModel.from_pretrained("Rostlab/prot_bert")

        for p in self.model.parameters():
            p.requires_grad = False

    def forward(self, seqs):
        seqs = [" ".join(list(s)) for s in seqs]
        inputs = self.tokenizer(
            seqs,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        out = self.model(**inputs)
        return out.last_hidden_state[:, 0, :]  # CLS

class HybridCPI(nn.Module):
    def __init__(self):
        super().__init__()
        self.gnn = CompoundGNN()
        self.protein = ProteinEncoder()

        fusion_dim = 128 + 2048 + 1024 + 20

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, batch):
        gnn_emb = self.gnn(batch.x, batch.edge_index, batch.batch)
        # after batching, fp and aac will have shape (batch_size, dim)
        fp = batch.fp
        aac = batch.aac
        prot_emb = self.protein(batch.protein_seq)

        return self.classifier(torch.cat([gnn_emb, fp, prot_emb, aac], dim=1)).squeeze()

# ===============================
# 6. LOSS
# ===============================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()

# ===============================
# 7. TRAIN & VAL FUNCTIONS
# ===============================
def train_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0
    all_probs = []
    all_targets = []

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        preds = model(batch)
        loss = loss_fn(preds, batch.y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        probs = torch.sigmoid(preds).detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        all_targets.extend(batch.y.cpu().numpy().tolist())

    avg_loss = total_loss / len(loader)
    try:
        auc = roc_auc_score(all_targets, all_probs)
    except:
        auc = None
    return avg_loss, auc

def validate_epoch(model, loader, loss_fn):
    model.eval()
    total_loss = 0
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            preds = model(batch)
            loss = loss_fn(preds, batch.y)

            total_loss += loss.item()
            probs = torch.sigmoid(preds).cpu().numpy()
            all_probs.extend(probs.tolist())
            all_targets.extend(batch.y.cpu().numpy().tolist())

    avg_loss = total_loss / len(loader)
    try:
        auc = roc_auc_score(all_targets, all_probs)
    except:
        auc = None
    return avg_loss, auc

# ===============================
# 8. MAIN
# ===============================
if __name__ == "__main__":
    # Set random seeds for reproducibility
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    data_path = "../dataset/b_cancer/original/data.txt"
    # data_path = "./sample_data.txt"

    # Load full dataset once
    full_dataset = CPIDataset(data_path)
    print(f"Loaded dataset with {len(full_dataset)} samples")

    # Split indices deterministically
    indices = list(range(len(full_dataset)))
    train_indices, temp_indices = train_test_split(indices, test_size=0.2, random_state=42, stratify=[full_dataset.samples[i][2] for i in indices])
    val_indices, test_indices = train_test_split(temp_indices, test_size=0.5, random_state=42, stratify=[full_dataset.samples[i][2] for i in temp_indices])

    print(f"Train: {len(train_indices)}, Val: {len(val_indices)}, Test: {len(test_indices)}")

    # Create subset datasets
    train_dataset = Subset(full_dataset, train_indices)
    val_dataset = Subset(full_dataset, val_indices)
    test_dataset = Subset(full_dataset, test_indices)

    # Create data loaders
    batch_size = 32
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Model, optimizer, loss
    model = HybridCPI().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    loss_fn = FocalLoss()

    # Training with early stopping
    num_epochs = 100
    patience = 10
    best_auc = 0
    patience_counter = 0

    for epoch in range(num_epochs):
        train_loss, train_auc = train_epoch(model, train_loader, optimizer, loss_fn)
        val_loss, val_auc = validate_epoch(model, val_loader, loss_fn)

        print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f}, Train AUC: {train_auc:.4f} | Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")

        # Save best model based on val AUC
        if val_auc is not None and val_auc > best_auc:
            best_auc = val_auc
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pt")
            print(f"  -> Saved best model with Val AUC: {best_auc:.4f}")
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    # Load best model and evaluate on test
    print("\nEvaluating on test set...")
    model.load_state_dict(torch.load("best_model.pt", map_location=device))
    test_loss, test_auc = validate_epoch(model, test_loader, loss_fn)

    # Compute additional metrics
    model.eval()
    all_probs = []
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            logits = model(batch)
            probs = torch.sigmoid(logits).cpu().numpy()
            y = batch.y.cpu().numpy()

            probs = probs.reshape(-1)
            preds = (probs >= 0.5).astype(int)

            all_probs.extend(probs.tolist())
            all_preds.extend(preds.tolist())
            all_targets.extend(y.reshape(-1).tolist())

    all_probs = np.array(all_probs)
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)

    acc = accuracy_score(all_targets, all_preds)
    prec = precision_score(all_targets, all_preds, zero_division=0)
    rec = recall_score(all_targets, all_preds, zero_division=0)

    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test AUC: {test_auc:.4f}")
    print(f"Test Accuracy: {acc:.4f}")
    print(f"Test Precision: {prec:.4f}")
    print(f"Test Recall: {rec:.4f}")

Using device: cpu
Loaded dataset with 17 samples
Train: 13, Val: 2, Test: 2


[16:21:52] DEPRECATION WARNING: please use MorganGenerator
[16:21:52] DEPRECATION WARNING: please use MorganGenerator
[16:21:52] DEPRECATION WARNING: please use MorganGenerator
[16:21:52] DEPRECATION WARNING: please use MorganGenerator
[16:21:52] DEPRECATION WARNING: please use MorganGenerator
[16:21:52] DEPRECATION WARNING: please use MorganGenerator
[16:21:52] DEPRECATION WARNING: please use MorganGenerator
[16:21:52] DEPRECATION WARNING: please use MorganGenerator
[16:21:52] DEPRECATION WARNING: please use MorganGenerator
[16:21:52] DEPRECATION WARNING: please use MorganGenerator
[16:21:52] DEPRECATION WARNING: please use MorganGenerator
[16:21:52] DEPRECATION WARNING: please use MorganGenerator
[16:21:52] DEPRECATION WARNING: please use MorganGenerator
[16:22:30] DEPRECATION WARNING: please use MorganGenerator
[16:22:30] DEPRECATION WARNING: please use MorganGenerator


Epoch   1 | Train Loss: 0.1295, Train AUC: 0.5714 | Val Loss: 0.1318, Val AUC: 0.0000


[16:22:37] DEPRECATION WARNING: please use MorganGenerator
[16:22:37] DEPRECATION WARNING: please use MorganGenerator
[16:22:37] DEPRECATION WARNING: please use MorganGenerator
[16:22:37] DEPRECATION WARNING: please use MorganGenerator
[16:22:37] DEPRECATION WARNING: please use MorganGenerator
[16:22:37] DEPRECATION WARNING: please use MorganGenerator
[16:22:37] DEPRECATION WARNING: please use MorganGenerator
[16:22:37] DEPRECATION WARNING: please use MorganGenerator
[16:22:37] DEPRECATION WARNING: please use MorganGenerator
[16:22:37] DEPRECATION WARNING: please use MorganGenerator
[16:22:37] DEPRECATION WARNING: please use MorganGenerator
[16:22:37] DEPRECATION WARNING: please use MorganGenerator
[16:22:37] DEPRECATION WARNING: please use MorganGenerator
[16:23:25] DEPRECATION WARNING: please use MorganGenerator
[16:23:25] DEPRECATION WARNING: please use MorganGenerator


Epoch   2 | Train Loss: 0.1285, Train AUC: 0.7143 | Val Loss: 0.1321, Val AUC: 0.0000


...

[16:25:14] DEPRECATION WARNING: please use MorganGenerator
[16:25:14] DEPRECATION WARNING: please use MorganGenerator
[16:25:14] DEPRECATION WARNING: please use MorganGenerator
[16:25:14] DEPRECATION WARNING: please use MorganGenerator
[16:25:14] DEPRECATION WARNING: please use MorganGenerator
[16:25:14] DEPRECATION WARNING: please use MorganGenerator
[16:25:14] DEPRECATION WARNING: please use MorganGenerator
[16:25:14] DEPRECATION WARNING: please use MorganGenerator
[16:25:14] DEPRECATION WARNING: please use MorganGenerator
[16:25:14] DEPRECATION WARNING: please use MorganGenerator
[16:25:14] DEPRECATION WARNING: please use MorganGenerator
[16:25:14] DEPRECATION WARNING: please use MorganGenerator
[16:25:14] DEPRECATION WARNING: please use MorganGenerator
[16:25:56] DEPRECATION WARNING: please use MorganGenerator
[16:25:56] DEPRECATION WARNING: please use MorganGenerator


Epoch   5 | Train Loss: 0.1220, Train AUC: 0.9286 | Val Loss: 0.1330, Val AUC: 0.0000


[16:26:01] DEPRECATION WARNING: please use MorganGenerator
[16:26:01] DEPRECATION WARNING: please use MorganGenerator
[16:26:01] DEPRECATION WARNING: please use MorganGenerator
[16:26:01] DEPRECATION WARNING: please use MorganGenerator
[16:26:01] DEPRECATION WARNING: please use MorganGenerator
[16:26:01] DEPRECATION WARNING: please use MorganGenerator
[16:26:01] DEPRECATION WARNING: please use MorganGenerator
[16:26:01] DEPRECATION WARNING: please use MorganGenerator
[16:26:01] DEPRECATION WARNING: please use MorganGenerator
[16:26:01] DEPRECATION WARNING: please use MorganGenerator
[16:26:01] DEPRECATION WARNING: please use MorganGenerator
[16:26:01] DEPRECATION WARNING: please use MorganGenerator
[16:26:01] DEPRECATION WARNING: please use MorganGenerator
[16:26:42] DEPRECATION WARNING: please use MorganGenerator
[16:26:42] DEPRECATION WARNING: please use MorganGenerator


Epoch   6 | Train Loss: 0.1200, Train AUC: 0.9524 | Val Loss: 0.1333, Val AUC: 0.0000


[16:26:47] DEPRECATION WARNING: please use MorganGenerator
[16:26:47] DEPRECATION WARNING: please use MorganGenerator
[16:26:47] DEPRECATION WARNING: please use MorganGenerator
[16:26:47] DEPRECATION WARNING: please use MorganGenerator
[16:26:47] DEPRECATION WARNING: please use MorganGenerator
[16:26:47] DEPRECATION WARNING: please use MorganGenerator
[16:26:47] DEPRECATION WARNING: please use MorganGenerator
[16:26:47] DEPRECATION WARNING: please use MorganGenerator
[16:26:47] DEPRECATION WARNING: please use MorganGenerator
[16:26:48] DEPRECATION WARNING: please use MorganGenerator
[16:26:48] DEPRECATION WARNING: please use MorganGenerator
[16:26:48] DEPRECATION WARNING: please use MorganGenerator
[16:26:48] DEPRECATION WARNING: please use MorganGenerator
